# 04 — Results Visualization

Loads the CSVs produced by `03_modeling.ipynb` (`cv_results.csv`, `predictions.csv`,
`lasso_coefficients.csv`) and `02_eda_featurization.ipynb` (`features.csv`), and produces the four
report figures, saved to `../figures/`.

## Summary

- **Part 2** queries only experimentally observed metals (`theoretical=False`, `band_gap == 0`) —
  real, ICSD-matched materials rather than every DFT-predicted candidate.
- **Part 3** deduplicates to the lowest-energy polymorph per unique composition, which matters
  specifically because MAGPIE features are compositional-only.
- **Parts 4–5** classify elemental family/period and take a stratified, capped sample across
  (family × crystal system) *before* the expensive per-material DOS lookups.
- **Part 6** extracts DOS(E_F) per material by interpolating the total DOS at the reported Fermi
  energy.
- **Part 8** compares ordinary K-fold CV against `GroupKFold`-by-family and `GroupKFold`-by-period —
  the key diagnostic for whether the Random Forest generalizes to unseen elemental chemistry or is
  simply interpolating within families it has already seen.
- **Part 10** adds a complementary, simpler diagnostic: a random 80/20 train/test split with a
  parity plot and a train-vs-test RMSE comparison, checking for plain overfitting independent of the
  elemental-generalization question Part 8 addresses.
- **Part 11** (Lasso) narrows the MAGPIE + crystal-system descriptor set down to the subset that
  actually carries predictive weight for DOS(E_F).
- **Part 12**'s crystal-system violin plot uses a log y-axis, since DOS(E_F) is strictly positive and
  heavily right-skewed.
- DOS(E_F) is a *screening* proxy for superconducting tendency, not a substitute for it: it captures
  one factor (electronic density of states) in the McMillan/Allen-Dynes electron-phonon coupling
  picture, but says nothing about phonon frequencies or the electron-phonon matrix element.
  High-scoring candidates from this pipeline are best treated as a shortlist for actual
  electron-phonon coupling (e.g. DFPT / EPW) calculations, not as final Tc predictions.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

os.makedirs("../figures", exist_ok=True)
print("Imports OK")


---
## Part 9 — Visualizing the CV comparison

The three MAE numbers from Part 8 (ordinary K-fold, GroupKFold-by-family, GroupKFold-by-period) are
the actual answer to "does this generalize," but as printed floats scattered across three cells
they're easy to under-weight. Plotting them side by side makes the generalization gap immediate.

In [ ]:
cv_results = pd.read_csv("../data/cv_results.csv")

plt.figure(figsize=(8, 5))
bars = plt.bar(cv_results["scheme"], cv_results["mean_mae"],
               yerr=cv_results["std_mae"], capsize=6,
               color=["#4C72B0", "#DD8452", "#DD8452"])
plt.ylabel("MAE  (states / eV / formula unit)")
plt.title("Does the model generalize to unseen elemental chemistry?")
for bar, val in zip(bars, cv_results["mean_mae"]):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{val:.2f}",
              ha="center", va="bottom")
plt.tight_layout()
plt.savefig("../figures/fig1.png", dpi=300)
plt.show()


---
## Part 10 (visualization) — Parity plot and train/test RMSE

A predicted-vs-actual parity plot for the held-out 80/20 split (Part 10 in `03_modeling.ipynb`),
alongside a direct train-vs-test RMSE comparison.

In [ ]:
predictions = pd.read_csv("../data/predictions.csv")

train = predictions[predictions["split"] == "train"]
test = predictions[predictions["split"] == "test"]

rmse_train = np.sqrt(mean_squared_error(train["y_true"], train["y_pred"]))
rmse_test = np.sqrt(mean_squared_error(test["y_true"], test["y_pred"]))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Parity plot: predicted vs actual, train and test overlaid ---
ax = axes[0]
lims = [0, predictions[["y_true", "y_pred"]].values.max() * 1.05]
ax.plot(lims, lims, "k--", linewidth=1, label="y = x")
ax.scatter(train["y_true"], train["y_pred"], alpha=0.3, s=15, label=f"Train (RMSE={rmse_train:.2f})")
ax.scatter(test["y_true"], test["y_pred"], alpha=0.5, s=15, label=f"Test (RMSE={rmse_test:.2f})")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Actual DOS(E_F)")
ax.set_ylabel("Predicted DOS(E_F)")
ax.set_title("Predicted vs. actual")
ax.legend()

# --- Train vs test RMSE bar chart ---
ax = axes[1]
bars = ax.bar(["Train", "Test"], [rmse_train, rmse_test], color=["#4C72B0", "#DD8452"])
for bar, val in zip(bars, [rmse_train, rmse_test]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{val:.2f}",
            ha="center", va="bottom")
ax.set_ylabel("RMSE  (states / eV / formula unit)")
ax.set_title("Train vs. test RMSE (80/20 split)")

plt.tight_layout()
plt.savefig("../figures/fig2.png", dpi=300)
plt.show()


---
## Part 11 (visualization) — Lasso feature importance

The top 20 MAGPIE / crystal-system descriptors by standardized Lasso coefficient magnitude.

In [ ]:
lasso_coefficients = pd.read_csv("../data/lasso_coefficients.csv").set_index("feature")["coefficient"]
top_n = lasso_coefficients.reindex(
    lasso_coefficients.abs().sort_values(ascending=False).index
).head(20).sort_values()   # ascending so barh reads largest-at-top

plt.figure(figsize=(9, 7))
colors = ["#4C72B0" if v > 0 else "#DD8452" for v in top_n.values]
plt.barh(top_n.index, top_n.values, color=colors)
plt.xlabel("Standardized Lasso coefficient")
plt.title("Top 20 MAGPIE / crystal-system features for DOS(E_F)")
plt.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.savefig("../figures/fig3.png", dpi=300)
plt.show()


---
## Part 12 (optional) — DOS(E_F) by crystal system

A quick visual check for whether particular crystal symmetries are systematically associated with
higher DOS(E_F) — a coarse, cheap signal for which structural families might be worth prioritizing
in a follow-up electron-phonon coupling screen.

In [ ]:
df_features = pd.read_csv("../data/features.csv")

order = df_features.groupby("crystal_system")["dos_ef"].median().sort_values(ascending=False).index

# DOS(E_F) is strictly positive and heavily right-skewed (a handful of large-DOS outliers,
# e.g. flat-band rare-earth compounds, alongside a bulk of modest values) - on a linear y-axis
# a violin plot like this collapses into a thin sliver near zero with a few needle-thin whiskers.
# Log-scaling the y-axis is what actually lets the shape comparison across crystal systems work.
plt.figure(figsize=(10, 6))
sns.violinplot(data=df_features, x="crystal_system", y="dos_ef", order=order, cut=0)
plt.ylabel("DOS(E_F)  (states / eV / formula unit, log scale)")
plt.xlabel("Crystal system")
plt.yscale("log")
plt.ylim(bottom=1e-0, top=1e2)  # avoid extreme outliers dominating the y-axis
plt.title("DOS(E_F) distribution by crystal system")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("../figures/fig4.png", dpi=300)
plt.show()
